In [1]:
import pandas as pd

In [4]:
import optuna

STORAGE = "sqlite:///optuna.db"

# список всех исследований (study_name)
print("Studies in storage:")
for s in optuna.study.get_all_study_summaries(storage=storage):
    print(f"- {s.study_name}, direction={s.direction}, trials={s.n_trials}")


Studies in storage:
- optuna/age, direction=1, trials=100
- optuna/alphabattle_small, direction=1, trials=100
- optuna/v2/age, direction=1, trials=10
- optuna/v3/age, direction=1, trials=1
- optuna/v4/age, direction=1, trials=1
- optuna/v5/age, direction=1, trials=1
- optuna/v6/age, direction=1, trials=1
- optuna/v20/age, direction=1, trials=16
- optuna/test/age, direction=1, trials=11
- optuna/v200/age, direction=1, trials=8
- optuna/test1/age, direction=1, trials=1
- optuna/test2/age, direction=1, trials=2
- optuna/test3/age, direction=1, trials=2
- optuna/test4/age, direction=1, trials=4
- optuna/test5/age, direction=1, trials=4
- optuna/test6/age, direction=1, trials=8
- optuna/v2/alphabattle_small, direction=1, trials=5
- optuna/v3/alphabattle_small, direction=1, trials=1
- optuna/v4/alphabattle_small, direction=1, trials=1
- optuna/v5/alphabattle_small, direction=1, trials=25
- optuna/v1/alphabattle_full, direction=1, trials=24


In [9]:
# --- базовая загрузка ---
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

STUDY   = "optuna/v200/age"              # имя вашего исследования
METRIC  = "loss"                 # столбец-метрика (Optuna кладёт целевую сюда)

study  = optuna.load_study(storage=STORAGE, study_name=STUDY)
trials = study.get_trials(deepcopy=False)

def trials_to_df(trials) -> pd.DataFrame:
    rows = []
    for t in trials:
        row = {
            "trial": t.number,
            "state": str(t.state),
            "value": t.value,
            "start": t.datetime_start,
            "end":   t.datetime_complete,
        }
        row["duration_sec"] = (
            (t.datetime_complete - t.datetime_start).total_seconds()
            if t.datetime_start and t.datetime_complete else np.nan
        )
        # расплющиваем параметры
        for k, v in t.params.items():
            row[f"param::{k}"] = v
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("trial").reset_index(drop=True)
    return df

df = trials_to_df(trials)
df_completed = df[df["state"] == "TrialState.COMPLETE"].copy()

print("Всего трейлов:", len(df), " | завершено:", len(df_completed))
display(df_completed.sort_values('value', ascending=False))


Всего трейлов: 8  | завершено: 3


,trial,state,value,start,end,duration_sec,param::optim.lr,param::optim.weight_decay,param::lr_scheduler.num_warmup_steps,param::training.ema,param::model.hidden_size,param::model.n_heads,param::model.n_blocks,param::model.cond_dim,param::model.dropout,param::noise.sigma_min,param::noise.sigma_max,param::training.sampling_eps,param::time_conditioning
3,3,TrialState.COMPLETE,2.24838,2025-09-10 18:39:04.619761,2025-09-10 21:18:59.300968,9594.681207,0.000515,9.402179e-03,500,0.999,128,8,12,128,0.259063,0.001743,23.374912,0.000234,True
6,6,TrialState.COMPLETE,1.81814,2025-09-11 05:59:05.192596,2025-09-11 11:11:40.837172,18755.644576,0.000011,3.017881e-03,100,0.999,128,8,24,128,0.016173,0.000182,32.858699,0.030149,True
4,4,TrialState.COMPLETE,1.80442,2025-09-10 21:18:59.324753,2025-09-11 05:58:47.920936,31188.596183,0.000013,1.188342e-11,200,0.999,512,4,16,64,0.165284,0.001722,31.974717,0.000519,True
